<a href="https://colab.research.google.com/github/manish7725/deeplearning/blob/main/Lecture%2003%20-%20Matrices%3A%20The%20Spreadsheet%20of%20Mathematics/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 03 — Matrices: The Spreadsheet of Mathematics · Laboratory

↩ **Theory:** [`blog.md`](<blog.md>) — read the lecture first. This notebook tests it.

## Step 1 — The Problem

The same four houses, and now three different valuations wanted for each:

| House | Rooms | Area (sq ft) | market | insurance | tax |
|---|---:|---:|---:|---:|---:|
| A | 2 | 800 | ? | ? | ? |
| B | 2 | 1200 | ? | ? | ? |
| C | 3 | 900 | ? | ? | ? |
| D | 4 | 1600 | ? | ? | ? |

That is **12 dot products**. Writing one line of mathematics per house does not scale to a
million houses — and we cannot differentiate a notation we cannot write down.

## Step 2 — Prediction

Commit before running. You check these in Step 10.

1. `X` has shape `(4, 2)` and `W` has shape `(2, 3)`. What shape is `X @ W`?
2. Which of these are legal: `X @ W`, `W @ X`, `X * W`?
3. `error` has shape `(4,)` and `w` has shape `(2,)`. Which expression gives a gradient with
   the right shape — `X @ error` or `X.T @ error`? Answer from the shapes alone.
4. For 2,000 houses with 50 features, is `X @ w` faster than a Python loop by about 2×, 10×,
   or more than 100×?

In [ ]:
# Step 3 — Intuition: the loop we are trying to get rid of.
import numpy as np
np.random.seed(0)

rooms  = np.array([2., 2., 3., 4.])
area   = np.array([800., 1200., 900., 1600.])
prices = np.array([9., 11., 11.5, 17.])

X = np.column_stack([rooms, area])     # the data matrix: rows are houses
w = np.array([2.0, 0.005])             # market-value weights
b = 1.0

print("X =\n", X, "\nshape:", X.shape)

# The loop: one dot product per house, exactly what section 1 complains about.
by_hand = []
for i in range(len(X)):
    by_hand.append(X[i] @ w + b)
print("\nloop result:  ", np.array(by_hand))

# The matrix form: the same mathematics, one expression, any number of houses.
print("matrix result:", X @ w + b)
assert np.allclose(by_hand, X @ w + b)
assert np.allclose(X @ w + b, prices)

## Step 4 — The Mathematics Under Test

$$\hat{\mathbf{y}} = X\mathbf{w} + b
\qquad
(XW)_{ij} = \sum_{k=1}^{d} X_{ik}W_{kj}
\qquad
(X^T)_{ij} = X_{ji}$$

and the rule that governs all of it:

$$(n \times d)\cdot(d \times m) \rightarrow (n \times m)$$

Step 5 checks every number the lecture claims.

In [ ]:
# Step 5 — Manual calculation: three valuations at once (blog sections 7-8).
W = np.array([[2.0, 0.0,   1.2  ],     # rooms     -> market, insurance, tax
              [0.005, 0.004, 0.003]])  # area      -> market, insurance, tax
biases = np.array([1.0, 0.0, 0.6])
print("X:", X.shape, " W:", W.shape, " ->  X @ W:", (X @ W).shape)

# One entry by hand: house A (row 1), tax (column 3), as section 8 writes it.
entry = X[0, 0] * W[0, 2] + X[0, 1] * W[1, 2]
print(f"\n(XW)_13 = {X[0,0]}*{W[0,2]} + {X[0,1]}*{W[1,2]} = {entry}")
assert entry == 4.8
assert (X @ W)[0, 2] == 4.8

Y_hat = X @ W + biases
print("\nY_hat =\n", Y_hat)
assert np.allclose(Y_hat[:, 0], prices)            # market column = Chapter 2's prices
assert np.allclose(Y_hat[:, 2], 0.6 * Y_hat[:, 0])  # tax really is 60% of market
print("\nmarket column matches Chapter 2; tax column is exactly 60% of market")

In [ ]:
# Step 5b — the shape rule as an error detector (blog sections 6 and 11).

# W @ X is undefined: inner dimensions 3 and 4 do not match.
try:
    W @ X
    raise SystemExit("should not get here")
except ValueError as e:
    print("W @ X refused, as it should:", e)

# Transpose swaps which axis gets summed over (section 10).
print("\nX  :", X.shape, "  X.T:", X.T.shape)
error = (X @ w + b) - prices          # shape (4,)

# Which expression can even produce a gradient shaped like w, i.e. (2,)?
try:
    X @ error
    raise SystemExit("should not get here")
except ValueError:
    print("X @ error   -> refused:  (4,2) against (4,) has no matching inner dimension")
print("X.T @ error -> shape", (X.T @ error).shape, "== shape of w", w.shape)
assert (X.T @ error).shape == w.shape

# Order matters, and the counterexample is tiny (section 9).
A = np.array([[1, 1], [0, 1]])
B = np.array([[1, 0], [1, 1]])
print("\nAB =\n", A @ B, "\nBA =\n", B @ A)
assert np.array_equal(A @ B, [[2, 1], [1, 1]])
assert np.array_equal(B @ A, [[1, 1], [1, 2]])
assert not np.array_equal(A @ B, B @ A)

In [ ]:
# Step 5c — the ladder: scalar, vector, matrix, tensor (blog section 12).
price      = np.array(9.0)                      # scalar
house      = np.array([2.0, 800.0])             # vector
dataset    = X                                  # matrix
photos     = np.zeros((4, 64, 64))              # 4 greyscale photos
colour     = np.zeros((4, 3, 64, 64))           # 4 colour photos

for name, t in [("scalar", price), ("vector", house), ("matrix", dataset),
                ("3-tensor", photos), ("4-tensor", colour)]:
    print(f"{name:>9}:  rank(ndim) = {t.ndim}   shape = {t.shape}")

assert price.ndim == 0 and house.ndim == 1 and dataset.ndim == 2
assert photos.ndim == 3 and colour.shape == (4, 3, 64, 64)

# The shape rule survives extra axes: the LAST axis is consumed,
# the leading (batch) axes ride along untouched.
batch  = np.zeros((32, 100, 64))    # 32 documents x 100 words x 64 numbers per word
layer  = np.zeros((64, 10))         # a layer with 10 outputs
out    = batch @ layer
print("\n(32,100,64) @ (64,10) ->", out.shape)
assert out.shape == (32, 100, 10)

# A matrix IS a rank-2 tensor; a vector IS a rank-1 tensor. One object, one algebra.

In [ ]:
# Step 6 — First implementation: matrix multiplication written out, exactly as
# the sigma in section 8 reads. Code mirrors the mathematics.

def matmul_explicit(A, B):
    n, d  = A.shape
    d2, m = B.shape
    assert d == d2, f"inner dimensions must match: {d} vs {d2}"
    C = np.zeros((n, m))
    for i in range(n):                 # each row of A
        for j in range(m):             # each column of B
            total = 0.0
            for k in range(d):         # walk along the row and down the column
                total += A[i, k] * B[k, j]
            C[i, j] = total
    return C

mine    = matmul_explicit(X, W)
library = X @ W
print("explicit:\n", mine)
print("\nnumpy:\n", library)
assert np.allclose(mine, library)
print("\nidentical. The library is faster, not different.")

In [ ]:
# Step 7 — Visualization: what "row meets column" actually looks like.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, M, title in [(axes[0], X, "X  (4 houses x 2 features)"),
                     (axes[1], W, "W  (2 features x 3 valuations)"),
                     (axes[2], X @ W + biases, "XW + b  (4 houses x 3 valuations)")]:
    # log scale keeps the 800s from washing out the 2s
    ax.imshow(np.log10(np.abs(M) + 1e-3), cmap="Blues")
    for (i, j), v in np.ndenumerate(M):
        ax.text(j, i, f"{v:g}", ha="center", va="center", fontsize=10)
    ax.set_title(title, fontsize=10)
    ax.set_xticks(range(M.shape[1])); ax.set_yticks(range(M.shape[0]))

axes[0].set_ylabel("houses")
plt.tight_layout(); plt.show()

print("Row i of X, met with column j of W, produced entry (i,j) on the right.")
print("The feature axis (length 2) was summed over and vanished.")

In [ ]:
# Step 8 — The controlled experiment: loop versus matrix form.
import time

def price_loop(Xb, wb):
    out = np.empty(len(Xb))
    for i in range(len(Xb)):
        total = 0.0
        for j in range(Xb.shape[1]):
            total += Xb[i, j] * wb[j]
        out[i] = total
    return out

rng = np.random.default_rng(0)
Xbig = rng.normal(size=(2000, 50))
wbig = rng.normal(size=50)

t0 = time.perf_counter(); slow = price_loop(Xbig, wbig); t1 = time.perf_counter()
t2 = time.perf_counter(); fast = Xbig @ wbig;            t3 = time.perf_counter()

assert np.allclose(slow, fast)        # same mathematics, to floating-point precision
print(f"loop:   {(t1-t0)*1000:9.3f} ms")
print(f"matrix: {(t3-t2)*1000:9.3f} ms")
print(f"speedup: {(t1-t0)/(t3-t2):,.0f}x  (exact number depends on your machine)")

In [ ]:
# Step 9 — Change exactly one variable: the number of houses.
print(f"{'n':>8} {'loop (ms)':>12} {'matrix (ms)':>14} {'speedup':>10}")
for n in [10, 100, 1000, 5000]:
    Xn = rng.normal(size=(n, 50))
    t0 = time.perf_counter(); price_loop(Xn, wbig); t1 = time.perf_counter()
    t2 = time.perf_counter(); Xn @ wbig;            t3 = time.perf_counter()
    print(f"{n:>8} {(t1-t0)*1000:12.3f} {(t3-t2)*1000:14.4f} {(t1-t0)/(t3-t2):9,.0f}x")

## Step 10 — Observe

Against your Step 2 predictions:

1. `X @ W` is `(4, 3)` — houses × valuations. The feature axis was consumed.
2. `X @ W` is legal; `W @ X` is refused (inner dimensions 3 and 4); `X * W` cannot even be
   attempted, since the shapes differ.
3. `X.T @ error` gives `(2,)`, matching `w`. `X @ error` is refused outright — **the shapes
   ruled out the wrong formula before any arithmetic happened.**
4. The matrix form typically wins by two or three orders of magnitude, and the gap *widens*
   with `n`.

Also worth noticing from Step 5c: a matrix is simply a rank-2 tensor, and pushing a
`(32, 100, 64)` batch through a `(64, 10)` layer follows the identical rule — last axis
consumed, leading axes untouched.

## Step 11 — Explain

**Why the shapes catch errors.** Every entry of a matrix product is a dot product, and a dot
product pairs slots one-to-one. If the two lengths differ, some slot has no partner and the
operation is undefined. That is the whole shape rule — and it is why a gradient must be
`X.T @ error`: only that arrangement sums over *houses* instead of over *features*.

**Why the matrix form is faster.** It is not a different algorithm — Step 6 proved the
numbers are identical. The Python loop interprets bytecode for every single multiplication
and chases pointers to boxed objects scattered through memory. `X @ w` hands the whole array
to compiled code that walks contiguous memory, uses several numbers per instruction, and
keeps the processor's cache full.

This is the entire hardware story of deep learning in one line. **A GPU is a machine built
to do Step 5's operation on enormous matrices**, which is why "expensive to train" means
"an extraordinary number of multiply-and-sum."

> But never let a fast implementation talk you out of checking the slow one. If the loop and
> the matrix form disagree, the fast one is wrong.

In [ ]:
# Step 12 — Challenges.

# LEVEL 2 (by hand first, then check):
# Compute X.T @ X. What shape is it? What is its top-left entry, and how does that
# relate to mean(rooms**2) from Chapter 2 section 10?
print("X.T @ X =\n", X.T @ X, "\nshape:", (X.T @ X).shape)
print("mean(rooms^2) =", np.mean(rooms ** 2), " and (X.T @ X)[0,0]/n =", (X.T @ X)[0, 0] / len(X))

# LEVEL 3 (Derive, then verify):
# Prove (AB).T == B.T @ A.T from the entry formula, then check it here.
# YOUR PROOF IN THE MARKDOWN CELL BELOW; the check is one line:
assert np.allclose((X @ W).T, W.T @ X.T)
print("\n(XW).T == W.T X.T  confirmed")

# LEVEL 4 (Investigate):
# Is computing all 3 valuations at once (X @ W) faster than 3 separate X @ w calls?
# Time both for a (5000, 50) matrix against a (50, 3) weight matrix, and explain.

# YOUR CODE HERE


# LEVEL 5 (Design):
# Add a 4th valuation, rental yield, that depends on area and on the neighbourhood
# encoding you designed in Chapter 2's Level 5. State the new shapes of X, W and Y_hat.
# Then: if 50,000 neighbourhoods each need a slot, what is the shape of W, how many
# numbers is that, and what does it tell you about why large models are large?

# YOUR CODE HERE

## Step 13 — Reflection

- [ ] I can state the shape rule in one sentence and say which axis disappears.
- [ ] I can explain why `X.T @ error` is the gradient and `X @ error` is not — from shapes alone.
- [ ] I wrote matrix multiplication as a triple loop and it matched NumPy exactly.
- [ ] I can give a tiny example where $AB \ne BA$.
- [ ] I can place scalar, vector, matrix and tensor on one ladder, and say what `rank` means
      on each — and that it means something *else* for a matrix in Chapter 4.
- [ ] I know why the vectorized form is faster without believing it is a different algorithm.

### The question this chapter leaves open

Look again at `X @ W`. Every house went in as a point with **two** coordinates and came out
as a point with **three**. The matrix did not merely store numbers — it *moved every point
from one space into another*.

So what does a matrix **do** to space? Does it stretch, rotate, flatten? Does applying $A$
then $B$ equal applying some single matrix $C$? And the question that decides the whole
architecture of a neural network: if you chain many matrices together, do you get something
genuinely new — or just another matrix?

➡️ **Next:** [Chapter 04 — A Matrix Can Transform Space](<../Lecture 04 - A Matrix Can Transform Space/blog.md>)